# Workshop 2.1: Fetching Historical Stock Data

Welcome to Stage 2! In the previous stages, we mastered Python programming foundations and explored pandas tabular operations. Now we step into real quantitative finance, where our models rely on actual market prices.

### The Anatomy of Market Price Bars

Before testing any quantitative trading strategy, we need historical price data. Financial markets capture trading activity through daily price bars known as **OHLCV**:
- **Open**: The price of the very first transaction recorded when the opening bell rings.
- **High**: The highest price transacted during regular trading hours.
- **Low**: The lowest price reached during the session.
- **Close**: The final traded price when the regular market session ends.
- **Volume**: The total number of shares traded during that day.

In this workshop, we will retrieve real market data using the open-source `yfinance` library and learn how to store our datasets locally using the high-performance Parquet format.

> **Key Takeaway**: Historical market observations are structured as OHLCV bars, giving us a complete record of price range and trading volume.

## Topic 1: Installing and Importing yfinance

To download market quotes directly into Python, we use the `yfinance` package, which interfaces with Yahoo Finance market feeds.

Let's install the package using pip. Let's see:

In [ ]:
# We install yfinance to fetch real market data:
!pip install yfinance

### Loading Core Quantitative Libraries

Now we import our standard quant stack:
- `yfinance as yf`: Provides data downloading functions.
- `pandas as pd`: Manages our time-series DataFrames.
- `numpy as np`: Powers vectorized mathematical arrays.

Let's import each package and verify that they load cleanly. Let's check:

In [2]:
# We import yfinance using the standard alias yf:
import yfinance as yf

print("yfinance loaded successfully as yf")

yfinance loaded successfully as yf


In [3]:
# We import pandas using alias pd:
import pandas as pd

print("pandas loaded successfully as pd")

pandas loaded successfully as pd


In [4]:
# We import numpy using alias np:
import numpy as np

print("numpy loaded successfully as np")

numpy loaded successfully as np


> **Key Takeaway**: We combine `yfinance` for data access with `pandas` and `numpy` for analysis.

---

## Topic 2: Downloading Historical Market Data

We retrieve price histories using `yf.download()`. This function takes a few essential parameters:
- **Ticker**: The market symbol, such as `"AAPL"` for Apple.
- **start**: The starting date of the historical window.
- **end**: The concluding date of the data window.
- **progress=False**: Silences download progress bars to keep our output tidy.

Let's fetch daily price records for Apple across the 2023 calendar year. Let's see:

In [5]:
# We download daily historical data for Apple across 2023:
aapl = yf.download("AAPL", start="2023-01-01", end="2024-01-01", progress=False)

print("First 5 rows of AAPL:")
print(aapl.head())

print("\nLast 5 rows of AAPL:")
print(aapl.tail())

First 5 rows of AAPL:
                  Open        High         Low       Close   Adj Close     Volume
Date                                                                             
2023-01-03  130.279999  130.899994  124.169998  125.070000  124.048241  112117500
2023-01-04  126.889999  128.660004  125.080002  126.360001  125.327690   89113600
2023-01-05  127.129997  127.769997  124.760002  125.019997  123.998650   87971200
2023-01-06  126.010002  130.289993  124.889999  129.619995  128.560944   87686600
2023-01-09  130.470001  133.410004  129.889999  130.149994  129.086594   70790800

Last 5 rows of AAPL:
                  Open        High         Low       Close   Adj Close    Volume
Date                                                                            
2023-12-22  195.179993  195.410004  192.970001  193.600006  193.090546  37122800
2023-12-26  193.610001  195.440002  193.500000  193.050003  192.542007  28919300
2023-12-27  192.490005  193.500000  191.089996  193.149994

> **Key Takeaway**: `yf.download(ticker, start, end)` fetches structured historical OHLCV data directly into a pandas DataFrame.

---

## Topic 3: Understanding Market Data Structure

Whenever new financial data arrives, we perform immediate diagnostic checks:
- `aapl.info()`: Inspects column names, non-null counts, and data types.
- `aapl.describe()`: Computes statistical summaries like mean and standard deviation.
- `aapl.shape`: Reports the total dimensions as `(rows, columns)`.

Notice that the DataFrame index is labeled `Date`, meaning each row represents a distinct trading day.

Let's run our inspection tools. Let's see:

In [6]:
# We inspect non-null counts and memory usage:
print("=== DataFrame Info ===")
aapl.info()

# We examine summary statistics for our numeric columns:
print("\n=== Summary Statistics ===")
print(aapl.describe())

# Let's check the total dimensions of our dataset:
print("\n=== Dimensions (rows, columns) ===")
print(aapl.shape)

=== DataFrame Info ===
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 250 entries, 2023-01-03 to 2023-12-29
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Open       250 non-null    float64
 1   High       250 non-null    float64
 2   Low        250 non-null    float64
 3   Close      250 non-null    float64
 4   Adj Close  250 non-null    float64
 5   Volume     250 non-null    int64  
dtypes: float64(5), int64(1)
memory usage: 13.7 KB

=== Summary Statistics ===
             Open        High         Low       Close   Adj Close        Volume
count  250.000000  250.000000  250.000000  250.000000  250.000000  2.500000e+02
mean   172.547400  174.200320  171.233400  172.814800  171.996162  5.911468e+07
std     17.734281   17.486241   17.747950   17.632111   17.653457  1.884350e+07
min    126.010002  127.769997  124.169998  125.019997  123.998650  2.404830e+07
25%    161.022503  162.597500  159.900002  161.739998  

> **Key Takeaway**: Diagnostic tools like `.info()` and `.describe()` give us immediate visibility into dataset quality.

---

## Topic 4: Date Handling: Timezone-Aware vs Timezone-Naive

Time-series dates in pandas can exist in two forms:
- **Timezone-aware**: Tied to a specific regional clock, such as `America/New_York`.
- **Timezone-naive**: Represents pure calendar dates without geographic offsets.

Timezone mismatches frequently trip up quantitative researchers. If you attempt to merge a timezone-aware price series with a timezone-naive macro series, pandas will halt with a `TypeError`.

To keep our datasets uniform and prevent alignment bugs, we strip timezone tags using `.tz_localize(None)`.

Let's inspect and standardize our index. Let's see:

In [7]:
# We inspect the index and its timezone status:
print("Current index:")
print(aapl.index)

print("\nTimezone attribute:")
print(aapl.index.tz)

# We strip timezone information to keep dates timezone-naive:
aapl.index = aapl.index.tz_localize(None)

print("
Updated index timezone attribute:")
print(aapl.index.tz)

Current index:
DatetimeIndex(['2023-01-03', '2023-01-04', '2023-01-05', '2023-01-06',
               '2023-01-09', '2023-01-10', '2023-01-11', '2023-01-12',
               '2023-01-13', '2023-01-17',
               ...
               '2023-12-18', '2023-12-19', '2023-12-20', '2023-12-21',
               '2023-12-22', '2023-12-26', '2023-12-27', '2023-12-28',
               '2023-12-29'],
              dtype='datetime64[ns]', name='Date', length=250, freq=None)

Timezone attribute:
None

Updated index timezone attribute:
None


> **Key Takeaway**: Stripping timezone metadata with `.tz_localize(None)` produces clean timezone-naive dates, preventing join conflicts down the road.

---

## Topic 5: Saving and Loading Data with Parquet

Downloading market data repeatedly across every backtest run is slow and risks hitting server rate limits. Professional quant workflows download data once and cache it immediately to local disk.

While beginners often reach for CSV, professional quantitative teams prefer **Parquet**:
- **Preserves Datatypes**: Columns retain their precise `float64` precision, and date indices stay formatted without manual parsing.
- **Speed and Compression**: Parquet reads and writes significantly faster than text files while consuming a fraction of the disk space.

Let's save our dataset to Parquet and verify that reloading it restores the exact same data. Let's check:

In [8]:
# We save our cleaned table to disk as a Parquet file:
aapl.to_parquet("aapl_2023.parquet")
print("Saved aapl_2023.parquet successfully")

# We read the Parquet file back into memory:
aapl_loaded = pd.read_parquet("aapl_2023.parquet")
print("Loaded aapl_2023.parquet successfully")

# We confirm the loaded table matches our original data:
print("\nDo the original and reloaded DataFrames match exactly?")
print(aapl.equals(aapl_loaded))

Saved aapl_2023.parquet successfully
Loaded aapl_2023.parquet successfully

Do the original and reloaded DataFrames match exactly?
True


> **Key Takeaway**: Parquet preserves exact datatypes and loads dramatically faster than CSV, making it the industry standard for financial caching.

---

## Topic 6: Selecting Specific Market Columns

Trading strategies often require only closing prices or trading volumes. We can isolate specific Series or sub-tables using bracket notation or date slices.

Let's practice extracting closing prices and slicing date intervals with `.loc`. Let's see:

In [9]:
# We extract a single column using bracket syntax:
print("=== Close Price (Series) ===")
print(aapl["Close"].head())

# We select multiple columns by passing a list:
print("\n=== Close and Volume (DataFrame) ===")
print(aapl[["Close", "Volume"]].head())

# We slice a specific date range using .loc:
print("\n=== First 3 days using .loc ===")
print(aapl.loc["2023-01-03":"2023-01-05", ["Close", "Volume"]])

=== Close Price (Series) ===
Date
2023-01-03    125.070000
2023-01-04    126.360001
2023-01-05    125.019997
2023-01-06    129.619995
2023-01-09    130.149994
Name: Close, dtype: float64

=== Close and Volume (DataFrame) ===
                 Close     Volume
Date                             
2023-01-03  125.070000  112117500
2023-01-04  126.360001   89113600
2023-01-05  125.019997   87971200
2023-01-06  129.619995   87686600
2023-01-09  130.149994   70790800

=== First 3 days using .loc ===
                 Close     Volume
Date                             
2023-01-03  125.070000  112117500
2023-01-04  126.360001   89113600
2023-01-05  125.019997   87971200


> **Key Takeaway**: We extract single market series with single brackets, multi-column subsets with double brackets, and date windows using `.loc`.

---

## Practice Time

Now it is your turn to download and cache historical financial datasets. Handling market data cleanly from the start ensures your downstream trading signals rest on a solid foundation.

---

### Challenge 1: Downloading Historical Equity Quotes

- Download historical daily data for Microsoft (`"MSFT"`) across the 2023 calendar year (`start="2023-01-01"`, `end="2024-01-01"`, `progress=False`).
- Display the first 5 rows of the downloaded data.

In [ ]:
# Challenge 1: Write your code below this line


# Expected: Prints the first 5 rows of MSFT data for 2023.


### Challenge 2: Stripping Timezones and Saving to Parquet

- Ensure your `MSFT` index is timezone-naive by applying `.tz_localize(None)`.
- Save the resulting DataFrame to a local Parquet file named `"msft_2023.parquet"`.

In [ ]:
# Challenge 2: Write your code below this line


# Expected: Saves 'msft_2023.parquet' to disk.


### Challenge 3: Verifying Cached Data Integrity

- Load `"msft_2023.parquet"` back into memory inside a variable named `msft_loaded` using `pd.read_parquet()`.
- Verify the reloaded dataset by printing its `.shape` dimensions.

In [ ]:
# Challenge 3: Write your code below this line


# Expected Output:
# (250, 6)


---

## Solutions Section

Great work completing these market data challenges! Establishing reliable local data caching is a critical first step in quantitative backtesting.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
msft = yf.download("MSFT", start="2023-01-01", end="2024-01-01", progress=False)
print("First 5 rows of MSFT:")
print(msft.head())
```

#### Solution for Challenge 2:
```python
msft.index = msft.index.tz_localize(None)
msft.to_parquet("msft_2023.parquet")
print("Saved msft_2023.parquet successfully.")
```

#### Solution for Challenge 3:
```python
msft_loaded = pd.read_parquet("msft_2023.parquet")
print("Loaded MSFT shape:")
print(msft_loaded.shape)
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [10]:
# Solution for Challenge 1:
msft = yf.download("MSFT", start="2023-01-01", end="2024-01-01", progress=False)
print("First 5 rows of MSFT:")
print(msft.head())

First 5 rows of MSFT:
                  Open        High         Low       Close   Adj Close    Volume
Date                                                                            
2023-01-03  243.080002  245.750000  237.399994  239.580002  235.772598  25740000
2023-01-04  232.279999  232.869995  225.960007  229.100006  225.459396  50623400
2023-01-05  227.199997  227.550003  221.759995  222.309998  218.777420  39585600
2023-01-06  223.000000  225.759995  219.350006  224.929993  221.355743  43613600
2023-01-09  226.449997  231.240005  226.410004  227.119995  223.510895  27369800


In [11]:
# Solution for Challenge 2:
msft.index = msft.index.tz_localize(None)
msft.to_parquet("msft_2023.parquet")
print("Saved msft_2023.parquet successfully.")

Saved msft_2023.parquet successfully.


In [12]:
# Solution for Challenge 3:
msft_loaded = pd.read_parquet("msft_2023.parquet")
print("Loaded MSFT shape:")
print(msft_loaded.shape)

Loaded MSFT shape:
(250, 6)
